In [1]:
!mkdir -p data_lake/bronze data_lake/silver/preprocessed_sales

In [2]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, when, to_date

# Initialize Spark Session
spark = SparkSession.builder.appName("DataLakeAssignment").getOrCreate()

In [3]:
# Read the Bronze dataset
df_bronze = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

# Display schema and first 5 rows
df_bronze.printSchema()
df_bronze.show(5)

# Show total initial record count
bronze_count = df_bronze.count()
print(f"Total records in Bronze layer: {bronze_count}")

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [14]:
from pyspark.sql.functions import col, trim, lower, expr

# 1. Remove exact duplicate rows
df_silver = df_bronze.dropDuplicates()

# 2. Handle missing values
df_silver = df_silver.dropna(subset=["order_id"])
df_silver = df_silver.fillna({"quantity": 0, "discount_percent": 0.0})

# 3. Trim extra spaces and standardize capitalization for text fields
for col_name in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    df_silver = df_silver.withColumn(col_name, lower(trim(col(col_name))))

# 4. Validate quantity, unit price, and discount percentage
df_silver = df_silver.filter(
    (col("quantity") >= 0) &
    (col("unit_price") >= 0) &
    (col("discount_percent") >= 0) & (col("discount_percent") <= 100)
)

# 5. Convert valid dates and safely handle invalid ones
# Using try_to_timestamp as suggested by your Spark environment, then casting to date
df_silver = df_silver.withColumn("order_date", expr("try_to_timestamp(order_date, 'yyyy-MM-dd')").cast("date"))

# Remove any rows where the date failed to parse (e.g., Feb 31st became null)
df_silver = df_silver.filter(col("order_date").isNotNull())

In [15]:
# Save the resulting DataFrame to the Silver folder as a single CSV file
# This satisfies the requirement to save processed data in the Silver layer in CSV format[cite: 1]
df_silver.coalesce(1).write.mode("overwrite").csv("data_lake/silver/preprocessed_sales", header=True)

# Verification Phase
# This satisfies the requirement to show row counts before and after preprocessing[cite: 1]
silver_count = df_silver.count()
print(f"Row count before preprocessing: {bronze_count}")
print(f"Row count after preprocessing: {silver_count}")

# Read the Silver data back with Spark and display a few rows[cite: 1]
df_silver_verify = spark.read.csv("data_lake/silver/preprocessed_sales", header=True, inferSchema=True)
df_silver_verify.show(5)

Row count before preprocessing: 1000
Row count after preprocessing: 966
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      sakshi joshi|            football|   sports|       2|       670|              30|     credit card| bengaluru|  karnataka|2025-06-06|   cancelled|
|  100333|      C0263|siddharth kulkarni|    programming book|    books|       4|       670|               0|     net banking|      pune|maharashtra|2025-05-08|   delivered|
|  100506|      C0228|     diya tripathi|competitive exam 

Removed exact duplicates, dropped rows with missing order_ids, filled missing numericals with 0, trimmed and standardized text fields to lowercase, filtered out negative quantities/prices and discounts over 100%, and standardized order_date to yyyy-MM-dd format while dropping impossible dates.